# HFA AI Training - Module 3: Vector Databases
## Working with ChromaDB

This notebook demonstrates the full ChromaDB workflow:
1. Creating a collection (like a table in a regular database)
2. Adding documents with embeddings and metadata
3. Querying by semantic similarity
4. Filtering results with metadata
5. Updating and deleting documents

ChromaDB is a lightweight, open-source vector database that runs entirely on your local machine. No server setup needed!

### Install Dependencies

In [ ]:
!pip install chromadb

### Section 1: Initialize ChromaDB

ChromaDB can run in two modes:
- **In-memory (ephemeral)** -- data disappears when script ends
- **Persistent** -- data saved to disk

We'll use in-memory for this demo. For production, you'd use:
```python
client = chromadb.PersistentClient(path="./my_vectordb")
```

In [ ]:
import chromadb

# ============================================================
# SECTION 1: Initialize ChromaDB
# ============================================================
# ChromaDB can run in two modes:
#   - In-memory (ephemeral) — data disappears when script ends
#   - Persistent — data saved to disk
#
# We'll use in-memory for this demo. For production, you'd use:
#   client = chromadb.PersistentClient(path="./my_vectordb")

print("=" * 60)
print("HFA AI Training - ChromaDB Example")
print("=" * 60)

# Create an in-memory ChromaDB client
# This is perfect for learning and experimentation
client = chromadb.Client()

print("\nChromaDB client initialized (in-memory mode)")

### Section 2: Create a Collection

A "collection" in ChromaDB is like a table in a regular database. Each collection stores documents, their embeddings, and metadata.

ChromaDB automatically generates embeddings for your documents using its built-in model -- you just provide the text!

In [ ]:
# ============================================================
# SECTION 2: Create a Collection
# ============================================================
# A "collection" in ChromaDB is like a table in a regular database.
# Each collection stores documents, their embeddings, and metadata.
#
# ChromaDB automatically generates embeddings for your documents
# using its built-in model — you just provide the text!

# Create (or get) a collection for real estate listings
# If the collection already exists, get_or_create_collection returns it
collection = client.get_or_create_collection(
    name="hawaii_listings",
    metadata={"description": "Hawaii real estate listings for semantic search"},
)

print(f"\nCreated collection: '{collection.name}'")
print(f"Current document count: {collection.count()}")

### Section 3: Add Documents

Each document needs:
- **id**: A unique identifier (string)
- **document**: The text content (ChromaDB auto-embeds this!)
- **metadata**: Key-value pairs for filtering (optional but useful)

ChromaDB handles the embedding automatically -- you just provide the raw text and it does the rest.

In [ ]:
# ============================================================
# SECTION 3: Add Documents
# ============================================================
# Each document needs:
#   - id:        A unique identifier (string)
#   - document:  The text content (ChromaDB auto-embeds this!)
#   - metadata:  Key-value pairs for filtering (optional but useful)
#
# ChromaDB handles the embedding automatically — you just
# provide the raw text and it does the rest.

print("\n--- Adding Documents ---\n")

# Real estate listings as documents
# Notice how each listing uses slightly different language
listings = {
    "ids": [
        "listing_001",
        "listing_002",
        "listing_003",
        "listing_004",
        "listing_005",
        "listing_006",
        "listing_007",
        "listing_008",
        "listing_009",
        "listing_010",
    ],
    "documents": [
        "Beautiful 3-bedroom oceanfront home in Kailua with panoramic views of the Pacific. "
        "Newly renovated kitchen, hardwood floors throughout. Steps from the beach.",

        "Charming 2-bed cottage in Haleiwa on the North Shore. Surf breaks within walking "
        "distance. Tropical garden, outdoor shower, laid-back island living at its finest.",

        "Luxury 4-bedroom penthouse in Waikiki with floor-to-ceiling windows overlooking "
        "Diamond Head. Full concierge service, rooftop pool, premium finishes throughout.",

        "Affordable 3-bed starter home in Ewa Beach. Family-friendly neighborhood, close "
        "to schools and shopping. Large backyard, perfect for growing families.",

        "Secluded mountain retreat in Manoa Valley. 2-bedroom home surrounded by lush "
        "tropical rainforest. Private trail access, serene waterfall nearby. Total privacy.",

        "Modern 1-bedroom condo in Kakaako with city and ocean views. Walking distance "
        "to restaurants, shops, and Ala Moana Beach Park. Ideal for young professionals.",

        "Historic plantation-style estate in Kailua-Kona on the Big Island. 5 bedrooms, "
        "wraparound lanai, mango and avocado trees. Rich with Hawaiian heritage.",

        "Beachfront studio in Maui with direct sand access. Perfect vacation rental or "
        "island getaway. Fully furnished, turnkey investment property.",

        "Spacious 4-bed family home in Mililani with mountain views. Excellent schools "
        "nearby, community pool, parks, and walking trails. Quiet, safe neighborhood.",

        "Eco-friendly off-grid tiny home on 2 acres in Puna, Big Island. Solar powered, "
        "rainwater catchment, organic garden. Sustainable Hawaiian living.",
    ],
    "metadatas": [
        {"beds": 3, "area": "Kailua", "island": "Oahu", "type": "house", "price_range": "high"},
        {"beds": 2, "area": "Haleiwa", "island": "Oahu", "type": "cottage", "price_range": "medium"},
        {"beds": 4, "area": "Waikiki", "island": "Oahu", "type": "penthouse", "price_range": "luxury"},
        {"beds": 3, "area": "Ewa Beach", "island": "Oahu", "type": "house", "price_range": "low"},
        {"beds": 2, "area": "Manoa", "island": "Oahu", "type": "house", "price_range": "high"},
        {"beds": 1, "area": "Kakaako", "island": "Oahu", "type": "condo", "price_range": "medium"},
        {"beds": 5, "area": "Kailua-Kona", "island": "Big Island", "type": "estate", "price_range": "high"},
        {"beds": 1, "area": "Maui", "island": "Maui", "type": "studio", "price_range": "medium"},
        {"beds": 4, "area": "Mililani", "island": "Oahu", "type": "house", "price_range": "medium"},
        {"beds": 1, "area": "Puna", "island": "Big Island", "type": "tiny home", "price_range": "low"},
    ],
}

# Add all documents to the collection in one batch
# ChromaDB automatically computes embeddings for each document
collection.add(
    ids=listings["ids"],
    documents=listings["documents"],
    metadatas=listings["metadatas"],
)

print(f"Added {collection.count()} listings to the collection.")
print("ChromaDB automatically generated embeddings for each listing!")

### Section 4: Basic Semantic Query

Now the magic: we can search using natural language, and ChromaDB finds the most semantically similar documents.

- **query_texts**: What you're looking for (in plain English)
- **n_results**: How many results to return

In [ ]:
# ============================================================
# SECTION 4: Basic Semantic Query
# ============================================================
# Now the magic: we can search using natural language,
# and ChromaDB finds the most semantically similar documents.
#
# query_texts: What you're looking for (in plain English)
# n_results:   How many results to return

print("\n\n--- QUERY 1: Basic Semantic Search ---")
print('Query: "beach house perfect for a family vacation"\n')

results = collection.query(
    query_texts=["beach house perfect for a family vacation"],
    n_results=3,
)

# Display results
# Results come back as lists of lists (to support batch queries)
for i in range(len(results["ids"][0])):
    doc_id = results["ids"][0][i]
    document = results["documents"][0][i]
    distance = results["distances"][0][i]
    metadata = results["metadatas"][0][i]

    # ChromaDB returns distances (lower = more similar)
    # Convert to a similarity score for easier understanding
    similarity = 1 - distance  # Rough conversion

    print(f"  Result #{i+1} (distance: {distance:.4f})")
    print(f"    ID:       {doc_id}")
    print(f"    Area:     {metadata['area']}, {metadata['island']}")
    print(f"    Type:     {metadata['type']} | Beds: {metadata['beds']}")
    print(f"    Snippet:  {document[:100]}...")
    print()

print("  Notice: It found beach-related family listings even though")
print("  the query words don't exactly match the listing text!")

### Section 5: Query with Metadata Filtering

The real power: combine semantic search with structured filters. "Find me something LIKE this (semantic) that ALSO meets these criteria (metadata filters)."

ChromaDB supports: `$eq`, `$ne`, `$gt`, `$gte`, `$lt`, `$lte`, `$in`, `$nin`, combined with `$and`, `$or`.

In [ ]:
# ============================================================
# SECTION 5: Query with Metadata Filtering
# ============================================================
# The real power: combine semantic search with structured filters.
# "Find me something LIKE this (semantic) that ALSO meets these
#  criteria (metadata filters)."
#
# ChromaDB supports: $eq, $ne, $gt, $gte, $lt, $lte, $in, $nin
# Combined with: $and, $or

print("\n--- QUERY 2: Semantic Search + Metadata Filtering ---")
print('Query: "quiet peaceful nature retreat"')
print('Filter: Oahu island only, at least 2 bedrooms\n')

results = collection.query(
    query_texts=["quiet peaceful nature retreat"],
    n_results=3,
    where={
        "$and": [
            {"island": {"$eq": "Oahu"}},
            {"beds": {"$gte": 2}},
        ]
    },
)

for i in range(len(results["ids"][0])):
    doc_id = results["ids"][0][i]
    document = results["documents"][0][i]
    distance = results["distances"][0][i]
    metadata = results["metadatas"][0][i]

    print(f"  Result #{i+1} (distance: {distance:.4f})")
    print(f"    ID:       {doc_id}")
    print(f"    Area:     {metadata['area']}, {metadata['island']}")
    print(f"    Type:     {metadata['type']} | Beds: {metadata['beds']}")
    print(f"    Snippet:  {document[:100]}...")
    print()

print("  The Manoa Valley retreat should rank highly — it matches the")
print("  'quiet peaceful nature' meaning AND the Oahu + 2-bed filters.")

### Section 6: Query with Price Range Filter

Let's filter for affordable options using the `$in` operator.

In [ ]:
# ============================================================
# SECTION 6: Query with Price Range Filter
# ============================================================
# Let's filter for affordable options

print("\n--- QUERY 3: Affordable Family Homes ---")
print('Query: "family home with good schools and outdoor space"')
print('Filter: Low or medium price range\n')

results = collection.query(
    query_texts=["family home with good schools and outdoor space"],
    n_results=3,
    where={
        "price_range": {"$in": ["low", "medium"]},
    },
)

for i in range(len(results["ids"][0])):
    doc_id = results["ids"][0][i]
    document = results["documents"][0][i]
    distance = results["distances"][0][i]
    metadata = results["metadatas"][0][i]

    print(f"  Result #{i+1} (distance: {distance:.4f})")
    print(f"    ID:       {doc_id}")
    print(f"    Area:     {metadata['area']}, {metadata['island']}")
    print(f"    Type:     {metadata['type']} | Beds: {metadata['beds']} | Price: {metadata['price_range']}")
    print(f"    Snippet:  {document[:100]}...")
    print()

### Section 7: Multiple Queries at Once

ChromaDB can process multiple queries in a single call. Each query gets its own set of results.

In [ ]:
# ============================================================
# SECTION 7: Multiple Queries at Once
# ============================================================
# ChromaDB can process multiple queries in a single call.
# Each query gets its own set of results.

print("\n--- QUERY 4: Batch Queries (Multiple at Once) ---\n")

queries = [
    "surfing lifestyle near the waves",
    "luxury living with premium amenities",
    "sustainable eco-friendly living off the grid",
]

results = collection.query(
    query_texts=queries,
    n_results=2,  # Top 2 for each query
)

for q_idx, query in enumerate(queries):
    print(f'  Query: "{query}"')
    for i in range(len(results["ids"][q_idx])):
        doc_id = results["ids"][q_idx][i]
        metadata = results["metadatas"][q_idx][i]
        distance = results["distances"][q_idx][i]
        print(f"    -> {metadata['area']} {metadata['type']} (distance: {distance:.4f})")
    print()

### Section 8: Update a Document

You can update documents, metadata, or both. Here we'll simulate a price drop by updating metadata.

In [ ]:
# ============================================================
# SECTION 8: Update a Document
# ============================================================
# You can update documents, metadata, or both

print("\n--- Updating a Document ---\n")

# Let's update the price range for a listing (price drop!)
collection.update(
    ids=["listing_004"],
    metadatas=[{"beds": 3, "area": "Ewa Beach", "island": "Oahu",
                "type": "house", "price_range": "low",
                "note": "PRICE REDUCED!"}],
)

# Verify the update by fetching the document directly
result = collection.get(ids=["listing_004"], include=["metadatas"])
print(f"  Updated listing_004 metadata: {result['metadatas'][0]}")

### Section 9: Delete a Document

Documents can be removed from a collection by ID (e.g., when a listing is sold).

In [ ]:
# ============================================================
# SECTION 9: Delete a Document
# ============================================================

print("\n--- Deleting a Document ---\n")

print(f"  Document count before delete: {collection.count()}")

# Remove a listing (maybe it sold!)
collection.delete(ids=["listing_008"])

print(f"  Document count after delete:  {collection.count()}")
print("  (listing_008 - Maui studio - has been removed)")

### Section 10: Peek at the Collection

The `peek` method shows a sample of documents in the collection -- useful for quick inspection.

In [ ]:
# ============================================================
# SECTION 10: Peek at the Collection
# ============================================================

print("\n--- Peeking at the Collection ---\n")

# Peek shows a sample of documents in the collection
peek = collection.peek(limit=3)
print(f"  Collection '{collection.name}' has {collection.count()} documents.")
print(f"  Sample IDs: {peek['ids']}")

### Key Takeaways

In [ ]:
# ============================================================
# SUMMARY
# ============================================================
print("\n" + "=" * 60)
print("KEY TAKEAWAYS:")
print("=" * 60)
print("""
  1. ChromaDB runs locally — no server or API key needed.
  2. Just add text documents — ChromaDB auto-generates embeddings.
  3. Query with natural language — results ranked by meaning similarity.
  4. Combine semantic search with metadata filters for precise results.
  5. Supports batch queries, updates, and deletes.
  6. Perfect for prototyping and learning before scaling to
     production vector databases like Pinecone or Weaviate.

  Next: See semantic_search.ipynb for a full real estate search demo!
""")